In [0]:
from pyspark.sql.types import DoubleType, DateType, IntegerType
from pyspark.sql.functions import col, regexp_replace, month, year, to_date, date_format
from delta.tables import DeltaTable

In [0]:
silver_df = (
    spark.read.table("lakehouse.bronze.precos_combustiveis")
    .withColumn(
        "valor_de_venda",
        regexp_replace(col("valor_de_venda"), ",", ".").cast(DoubleType())
    ).withColumn(
        "data_da_coleta",
        to_date(col("data_da_coleta"))
    ).withColumn(
        "mes_da_coleta",
        month(col("data_da_coleta"))
    ).withColumn(
        "ano_da_coleta",
        year(col("data_da_coleta"))
    ).withColumn(
    "unidade_de_medida",
    regexp_replace(col("unidade_de_medida"), r"R\$\s*/\s*", "")
    ).select(
        'regiao_sigla', 'estado_sigla', 'municipio', 'revenda', 'data_da_coleta', 'mes_da_coleta', 'ano_da_coleta', 'produto', 'valor_de_venda', 'unidade_de_medida', 'bandeira'
    )
)

In [0]:
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("lakehouse.silver.precos_combustiveis")